<a href="https://colab.research.google.com/github/Samy-Annasri/backcasting-for-adversarial-attacks-on-time-series/blob/main/BATS_Google_Stock.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## IMPORT

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import numpy as np
import random
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
seed = 999
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(seed)

In [ ]:
# just some code for testing the import
'''
import sys

if 'prepare_stock_dataset' in globals():
    del globals()['prepare_stock_dataset']

module_name = 'utils.setup_google_stock_dataset'
if module_name in sys.modules:
    del sys.modules[module_name]
'''

## STEP 1: PREP DATA

In [ ]:
google_data = pd.read_csv("data/HistoricalData_1747091015337.csv")
google_data['Date'] = pd.to_datetime(google_data['Date'])
google_data = google_data.sort_values(by='Date')
display(google_data)

In [ ]:
for col in ['Close/Last', 'Open', 'High', 'Low']:
    google_data[col] = google_data[col].replace('[\$,]', '', regex=True).astype(float)

In [ ]:
from utils.setup_google_stock_dataset import prepare_stock_dataset
sequence_length = 30
result = prepare_stock_dataset(google_data)

train_loader = result['train_loader']
test_loader = result['test_loader']
train_size = result['train_size']
min_max = result['min_max']
dates = result['dates']
price_min, price_max = min_max['Close/Last']
print(dates)

## STEP 2: TRAIN NORMAL LSTM

In [ ]:
from utils.train_model import train_model
from models.lstm import SimpleLSTM
model_google = SimpleLSTM(input_size=5, hidden_size=64, output_size=1, num_layers=2)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model_google.parameters(), lr=0.001)
num_epochs = 30
train_model(model_google, loss_fn, optimizer, num_epochs, train_loader)

In [ ]:
from utils.google_eval import evaluate_model_google
results = evaluate_model_google(model_google, test_loader, dates, train_size)

real_values = results['real_values']
predicted_values = results['predicted_values']
test_dates = results['test_dates']

In [ ]:
# Computes scalar similarity (cosine similarity) between true and predicted values.
# Higher values indicate that adversarial predictions remain directionally aligned
# with the true values, suggesting stealthy and rational attacks!
def scalar_similarity(y_true, y_pred):
    numerator = np.dot(y_true, y_pred)
    denominator = np.linalg.norm(y_true) * np.linalg.norm(y_pred)
    if denominator == 0:
        return 0.0
    return numerator / denominator

In [ ]:
# Creation of the tab result for plotting adversial attack result
models = ['LSTM','RNN','GRU']
metrics = ['MAE', "RMSE", 'SIM']

row_index = pd.MultiIndex.from_product([models, metrics], names=['Model', 'Metric'])

attacks = ['NA','PAST','REV']
epsilons = {
    'NA': [0],
    'PAST':[0],
    'REV':[0.01, 0.02, 0.03, 0.04, 0.05, 0.075, 0.1, 0.2],
}

col_tuples = []
for atk, eps_list in epsilons.items():
    for eps in eps_list:
        col_tuples.append((atk, f"{eps:.2f}"))

col_index = pd.MultiIndex.from_tuples(col_tuples, names=['Attack', 'ε'])

res_tab = pd.DataFrame(index=row_index, columns=col_index, dtype=float)

print(res_tab)

In [ ]:
from utils.log_and_plot import log_and_plot_predictions
# Denormalize
true_values_denorm = real_values * (price_max - price_min) + price_min
predictions_denorm = predicted_values * (price_max - price_min) + price_min

log_and_plot_predictions(
    true_values=true_values_denorm,
    predictions=predictions_denorm,
    test_dates=test_dates,
    true_values_rolling=None,
    predictions_rolling=None,
    model_name='LSTM',
    attack_name='NA',
    epsilon=0.00,
    res_tab=res_tab,
    similarity_fn=scalar_similarity,
    google=True,
    save_path=f"results/SEED_{seed}/plots/NA/0",
    save_png=True
)

In [ ]:
display(res_tab)

## STEP 3: REVERSE DATA



In [ ]:
from utils.setup_google_stock_dataset import prepare_stock_dataset
google_data_reversed = google_data
for col in ['Close/Last', 'Open', 'High', 'Low']:
    google_data_reversed[col] = google_data_reversed[col].replace('[\$,]', '', regex=True).astype(float)
result_rev = prepare_stock_dataset(google_data_reversed,reverse=True)
train_loader_rev = result_rev['train_loader']
test_loader_rev = result_rev['test_loader']
train_size_rev = result['train_size']
min_max_rev = result['min_max']
dates_rev = result_rev['dates']
price_min_rev, price_max_rev = min_max['Close/Last']

## STEP 4: TRAIN REVERSE LSTM

In [ ]:
model_google_rev = SimpleLSTM(input_size=5, hidden_size=64, output_size=1, num_layers=2)
optimizer_rev = torch.optim.Adam(model_google_rev.parameters(), lr=0.001)

train_model(model_google_rev, loss_fn, optimizer_rev, num_epochs, train_loader_rev)

In [ ]:
results_rev = evaluate_model_google(model_google_rev, test_loader_rev, dates_rev, train_size_rev)

real_values_rev = results_rev['real_values']
predicted_values_rev = results_rev['predicted_values']
test_dates_rev = results_rev['test_dates']

min_len = min(len(test_dates_rev), len(real_values_rev), len(predicted_values_rev))
true_values_denorm_rev = (real_values_rev * (price_max_rev - price_min_rev) + price_min_rev)[:min_len]
predictions_denorm_rev = (predicted_values_rev * (price_max_rev - price_min_rev) + price_min_rev)[:min_len]
test_dates_rev = test_dates_rev[:min_len]

log_and_plot_predictions(
    true_values=true_values_denorm_rev,
    predictions=predictions_denorm_rev,
    test_dates=test_dates_rev,
    true_values_rolling=None,
    predictions_rolling=None,
    model_name='LSTM',
    attack_name='PAST',
    epsilon=0.00,
    res_tab=res_tab,
    similarity_fn=scalar_similarity,
    google=True,
    reverse=True,
    save_path=f"results/SEED_{seed}/plots/PAST/0",
    save_png=True
)

## plot for the paper

In [ ]:
min_len = min(len(dates), len(google_data['Close/Last']))
dates_trimmed = dates[:min_len]
close_trimmed = google_data['Close/Last'][:min_len]

plt.figure(figsize=(8, 4))
plt.plot(dates_trimmed, close_trimmed, label='Original')
plt.title("Stock Price")
plt.xlabel("Date")
plt.ylabel("Stock Price")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
min_len = min(len(dates_rev), len(google_data_reversed['Close/Last']))
dates_trimmed = dates_rev[:min_len]
close_trimmed = google_data_reversed['Close/Last'][:min_len]

plt.figure(figsize=(8, 4))
plt.gca().invert_xaxis()
plt.plot(dates_trimmed, close_trimmed, label='Reverse Values', color='orange')
plt.xlabel('Date')
plt.ylabel('Stock Price')
plt.title('Stock Price reverse')
plt.legend()
plt.tight_layout()
plt.show()


# STEP 5: ATTACK FGSM NO AUTOREGRESSIVELY (FIRST ATTEMPT)

In [ ]:
from attack.rev import reverse_forecast_attack

epsilons = [0.01, 0.02, 0.03, 0.04, 0.05, 0.075, 0.1, 0.2]
results = reverse_forecast_attack(
    model_google_rev, model_google,
    test_loader_rev, test_loader,
    epsilons,
    price_min, price_max
)

for esp, (true_vals, preds) in results.items():
  log_and_plot_predictions(
      true_values=true_vals,
      predictions=preds,
      test_dates=test_dates,
      true_values_rolling=None,
      predictions_rolling=None,
      model_name='LSTM',
      attack_name='REV',
      epsilon=esp,
      res_tab=res_tab,
      similarity_fn=scalar_similarity,
      google=True,
      save_path=f"results/SEED_{seed}/plots/REV/{esp}",
      save_png=True
  )

  if esp == 0.03:
    min_len = min(len(test_dates), len(true_vals), len(predictions_denorm))
    plt.figure(figsize=(8, 6))

    plt.plot(test_dates[:min_len], predictions_denorm[:min_len],
             label='Base Prediction', color='green', linewidth=1.8)

    plt.plot(test_dates[:min_len], preds[:min_len],
             label=f'Adversarial Prediction (ε = {esp})', color='red', linewidth=1.8)

    plt.title(f'Comparison: Base vs Adversarial Prediction (ε = {esp})', fontsize=14)
    plt.xlabel('Date', fontsize=12)
    plt.ylabel('Stock Price', fontsize=12)

    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend(fontsize=10)
    plt.tight_layout()
    plt.grid(False)
    plt.show()

In [ ]:
esp = 0.03
true_vals_rev, preds_rev = results[esp]

min_len = min(len(test_dates), len(true_vals_rev))
plt.figure(figsize=(8, 4))

plt.plot(test_dates[:min_len], preds_rev[:min_len],
         label=f'Adversarial Prediction (Reversed Order)',
         color='orange', linewidth=1.8)

plt.gca().invert_xaxis()
plt.title(f'Adversarial Prediction (Reversed Order)', fontsize=14)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Stock Price', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=10)
plt.tight_layout()
plt.grid(False)
plt.show()

preds_normal_order = preds_rev[::-1]
dates_normal_order = test_dates[:len(preds_normal_order)][::-1]

plt.figure(figsize=(8, 4))
plt.plot(dates_normal_order, preds_normal_order,
         label=f'Adversarial Prediction',
         color='red', linewidth=1.8)

plt.title(f'Adversarial Prediction', fontsize=14)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Stock Price', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=10)
plt.tight_layout()
plt.grid(False)
plt.show()


In [ ]:
display(res_tab)

## STEP 5 FOR OTHER MODELS (RNN,CNN)

In [ ]:
from utils.train_model import train_model
from models.rnn import SimpleRNN
model_google_rnn = SimpleRNN(input_size=5, hidden_size=64, output_size=1, num_layers=2)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model_google_rnn.parameters(), lr=0.001)
num_epochs = 30
train_model(model_google_rnn, loss_fn, optimizer, num_epochs, train_loader)

In [ ]:
results_rnn = evaluate_model_google(model_google_rnn, test_loader, dates, train_size)

real_values_rnn = results_rnn['real_values']
predicted_values_rnn = results_rnn['predicted_values']
test_dates = results_rnn['test_dates']

true_values_denorm_rnn = real_values_rnn * (price_max - price_min) + price_min
predictions_denorm_rnn = predicted_values_rnn * (price_max - price_min) + price_min

log_and_plot_predictions(
    true_values=true_values_denorm_rnn,
    predictions=predictions_denorm_rnn,
    test_dates=test_dates,
    true_values_rolling=None,
    predictions_rolling=None,
    model_name='RNN',
    attack_name='NA',
    epsilon=0.00,
    res_tab=res_tab,
    similarity_fn=scalar_similarity,
    google=True,
    save_path=f"results/SEED_{seed}/plots/NA/0",
    save_png=True
)

In [ ]:
model_google_rnn_rev = SimpleRNN(input_size=5, hidden_size=64, output_size=1, num_layers=2)
optimizer_rev = torch.optim.Adam(model_google_rnn_rev.parameters(), lr=0.001)

train_model(model_google_rnn_rev, loss_fn, optimizer_rev, num_epochs, train_loader_rev)

In [ ]:
results_rnn_rev = evaluate_model_google(model_google_rnn_rev, test_loader_rev, dates_rev, train_size_rev)

real_values_rnn_rev = results_rnn_rev['real_values']
predicted_values_rnn_rev = results_rnn_rev['predicted_values']
test_dates_rev = results_rnn_rev['test_dates']

min_len = min(len(test_dates_rev), len(real_values_rnn_rev), len(predicted_values_rnn_rev))
true_values_denorm_rnn_rev = (real_values_rnn_rev * (price_max_rev - price_min_rev) + price_min_rev)[:min_len]
predictions_denorm_rnn_rev = (predicted_values_rnn_rev * (price_max_rev - price_min_rev) + price_min_rev)[:min_len]
test_dates_rev = test_dates_rev[:min_len]

log_and_plot_predictions(
    true_values=true_values_denorm_rnn_rev,
    predictions=predictions_denorm_rnn_rev,
    test_dates=test_dates_rev,
    true_values_rolling=None,
    predictions_rolling=None,
    model_name='RNN',
    attack_name='PAST',
    epsilon=0.00,
    res_tab=res_tab,
    similarity_fn=scalar_similarity,
    google=True,
    reverse=True,
    save_path=f"results/SEED_{seed}/plots/PAST/0",
    save_png=True
)

In [ ]:
from attack.rev import reverse_forecast_attack

results_rev_rnn = reverse_forecast_attack(
    model_google_rnn_rev, model_google_rnn,
    test_loader_rev, test_loader,
    epsilons,
    price_min, price_max
)

for esp, (true_vals, preds) in results_rev_rnn.items():
  log_and_plot_predictions(
      true_values=true_vals,
      predictions=preds,
      test_dates=test_dates,
      true_values_rolling=None,
      predictions_rolling=None,
      model_name='RNN',
      attack_name='REV',
      epsilon=esp,
      res_tab=res_tab,
      similarity_fn=scalar_similarity,
      google=True,
      save_path=f"results/SEED_{seed}/plots/REV/{esp}",
      save_png=True
  )

In [ ]:
display(res_tab)

In [ ]:
from models.gru import SimpleGRU
model_google_gru = SimpleGRU(input_size=5, hidden_size=64, output_size=1, num_layers=2)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model_google_gru.parameters(), lr=0.001)
num_epochs = 30
train_model(model_google_gru, loss_fn, optimizer, num_epochs, train_loader)

In [ ]:
results_gru = evaluate_model_google(model_google_gru, test_loader, dates, train_size)

real_values_gru = results_gru['real_values']
predicted_values_gru = results_gru['predicted_values']
test_dates = results_gru['test_dates']

true_values_denorm_gru = real_values_gru * (price_max - price_min) + price_min
predictions_denorm_gru = predicted_values_gru * (price_max - price_min) + price_min

log_and_plot_predictions(
    true_values=true_values_denorm_gru,
    predictions=predictions_denorm_gru,
    test_dates=test_dates,
    true_values_rolling=None,
    predictions_rolling=None,
    model_name='GRU',
    attack_name='NA',
    epsilon=0.00,
    res_tab=res_tab,
    similarity_fn=scalar_similarity,
    google=True,
    save_path=f"results/SEED_{seed}/plots/NA/0",
    save_png=True
)

In [ ]:
model_google_gru_rev = SimpleGRU(input_size=5, hidden_size=64, output_size=1, num_layers=2)
optimizer_rev = torch.optim.Adam(model_google_gru_rev.parameters(), lr=0.001)

train_model(model_google_gru_rev, loss_fn, optimizer_rev, num_epochs, train_loader_rev)

In [ ]:
results_gru_rev = evaluate_model_google(model_google_gru_rev, test_loader_rev, dates_rev, train_size_rev)

real_values_gru_rev = results_gru_rev['real_values']
predicted_values_gru_rev = results_gru_rev['predicted_values']
test_dates_rev = results_gru_rev['test_dates']

min_len = min(len(test_dates_rev), len(real_values_gru_rev), len(predicted_values_gru_rev))
true_values_denorm_gru_rev = (real_values_gru_rev * (price_max_rev - price_min_rev) + price_min_rev)[:min_len]
predictions_denorm_gru_rev = (predicted_values_gru_rev * (price_max_rev - price_min_rev) + price_min_rev)[:min_len]
test_dates_rev = test_dates_rev[:min_len]

log_and_plot_predictions(
    true_values=true_values_denorm_gru_rev,
    predictions=predictions_denorm_gru_rev,
    test_dates=test_dates_rev,
    true_values_rolling=None,
    predictions_rolling=None,
    model_name='GRU',
    attack_name='PAST',
    epsilon=0.00,
    res_tab=res_tab,
    similarity_fn=scalar_similarity,
    google=True,
    reverse=True,
    save_path=f"results/SEED_{seed}/plots/PAST/0",
    save_png=True
)

In [ ]:
results_rev_gru = reverse_forecast_attack(
    model_google_gru_rev, model_google_gru,
    test_loader_rev, test_loader,
    epsilons,
    price_min, price_max
)

for esp, (true_vals, preds) in results_rev_gru.items():
  log_and_plot_predictions(
      true_values=true_vals,
      predictions=preds,
      test_dates=test_dates,
      true_values_rolling=None,
      predictions_rolling=None,
      model_name='GRU',
      attack_name='REV',
      epsilon=esp,
      res_tab=res_tab,
      similarity_fn=scalar_similarity,
      google=True,
      save_path=f"results/SEED_{seed}/plots/REV/{esp}",
      save_png=True
  )

In [ ]:
display(res_tab)

# CLASIC FGSM JUST FOR THE EXPERIMENTATION

In [ ]:
from attack.fgsm import fgsm_attack

In [ ]:
# Creation of the tab result for plotting adversial attack result
models = ['LSTM','RNN','GRU']
metrics = ['MAE', "RMSE", 'SIM']

row_index = pd.MultiIndex.from_product([models, metrics], names=['Model', 'Metric'])

attacks = ['FGSM']
epsilons_attack = {
    'FGSM':[0.01, 0.02, 0.03, 0.04, 0.05, 0.075, 0.1, 0.2],
}

col_tuples = []
for atk, eps_list in epsilons_attack.items():
    for eps in eps_list:
        col_tuples.append((atk, f"{eps:.2f}"))

col_index = pd.MultiIndex.from_tuples(col_tuples, names=['Attack', 'ε'])

res_tab_fgsm = pd.DataFrame(index=row_index, columns=col_index, dtype=float)

print(res_tab_fgsm)

In [ ]:
epsilons_fgsm = [0.01, 0.02, 0.03, 0.04, 0.05, 0.075, 0.1, 0.2]

for eps in epsilons_fgsm:
    true_vals_fgsm, preds_fgsm = fgsm_attack(
        model_google_gru, test_loader, loss_fn, eps, price_min, price_max
    )

    log_and_plot_predictions(
        true_values=true_vals_fgsm,
        predictions=preds_fgsm,
        test_dates=test_dates,
        true_values_rolling=None,
        predictions_rolling=None,
        model_name='GRU',
        attack_name='FGSM',
        epsilon=eps,
        res_tab=res_tab_fgsm,
        similarity_fn=scalar_similarity,
        google=True,
        save_path=f"results/SEED_{seed}/plots/FGSM/{eps}",
        save_png=True
    )

    true_vals_fgsm, preds_fgsm = fgsm_attack(
        model_google, test_loader, loss_fn, eps, price_min, price_max
    )

    log_and_plot_predictions(
        true_values=true_vals_fgsm,
        predictions=preds_fgsm,
        test_dates=test_dates,
        true_values_rolling=None,
        predictions_rolling=None,
        model_name='LSTM',
        attack_name='FGSM',
        epsilon=eps,
        res_tab=res_tab_fgsm,
        similarity_fn=scalar_similarity,
        google=True,
        save_path=f"results/SEED_{seed}/plots/FGSM/{eps}",
        save_png=True
    )

    true_vals_fgsm, preds_fgsm = fgsm_attack(
        model_google_rnn, test_loader, loss_fn, eps, price_min, price_max
    )

    log_and_plot_predictions(
        true_values=true_vals_fgsm,
        predictions=preds_fgsm,
        test_dates=test_dates,
        true_values_rolling=None,
        predictions_rolling=None,
        model_name='RNN',
        attack_name='FGSM',
        epsilon=eps,
        res_tab=res_tab_fgsm,
        similarity_fn=scalar_similarity,
        google=True,
        save_path=f"results/SEED_{seed}/plots/FGSM/{eps}",
        save_png=True
    )


In [ ]:
display(res_tab)

In [ ]:
display(res_tab_fgsm)

Résult interpretation :


# Simple test using model_rev != model

In [ ]:
# I try to attaque model LSTM with GRU_rev to see what append (and GRU with LSTM,etc...)
rev_attacks = [
    {
        "model_rev": model_google_gru_rev,
        "model_target": model_google,
        "model_name": "LSTM"
    },
    {
        "model_rev": model_google_gru_rev,
        "model_target": model_google_rnn,
        "model_name": "RNN"
    },
    {
        "model_rev": model_google_rnn_rev,
        "model_target": model_google_gru,
        "model_name": "GRU"
    }
]

for attack in rev_attacks:
    results = reverse_forecast_attack(
        model_rev=attack["model_rev"],
        model_normal=attack["model_target"],
        test_loader_rev=test_loader_rev,
        test_loader=test_loader,
        epsilons=epsilons,
        price_min=price_min,
        price_max=price_max
    )

    for eps, (true_vals, preds) in results.items():
        log_and_plot_predictions(
            true_values=true_vals,
            predictions=preds,
            test_dates=test_dates,
            true_values_rolling=None,
            predictions_rolling=None,
            model_name=attack["model_name"],
            attack_name="REV_NO_EQUAL",
            epsilon=eps,
            res_tab=res_tab,
            similarity_fn=scalar_similarity,
            google=True,
            save_path=f"results/SEED_{seed}/plots/REV_NO_EQUAL/{eps}",
            save_png=True
        )

In [ ]:
display(res_tab)

We observe something really interesting :
* The more accurate the proxy model is, the lower the resulting MAE tends to be. In other words, when the proxy model is accurate, the attack becomes more stealthy, but slightly less aggressive in terms of prediction error.
* This is actually a good trade-off in time series attacks, where the objective is often to maximize disruption while remaining undetectable.
* That's also why, when we use a less accurate proxy (e.g., LSTM instead of GRU or RNN), the attack can result in a higher MAE meaning it's more aggressive but at the cost of being visually detectable.
* This is clearly visible in the plots: the more aggressive attacks often generate abnormal prediction curves, making the anomaly obvious.

* Also i don't know for the moment the real impact of using another model for proxy than the model we use for prediction? for exemple even when the proxy models achieve similar reverse prediction performance (i.e., same MAE), the resulting adversarial impact on the target model can vary significantly.
In particular, GRU-based proxy models tend to produce more effective attacks than RNNs, despite comparable accuracy.

# TEST WITH BIM

In [ ]:
from attack.rev_bim import reverse_forecast_attack_bim

In [ ]:
# Creation of the tab result for plotting adversial attack result
models = ['LSTM','RNN','GRU']
metrics = ['MAE', "RMSE", 'SIM']

row_index = pd.MultiIndex.from_product([models, metrics], names=['Model', 'Metric'])

attacks = ['REV_BIM']
epsilons = {
    'REV_BIM':[0.01,0.02,0.03,0.04, 0.05, 0.075, 0.1, 0.2],
}

col_tuples = []
for atk, eps_list in epsilons.items():
    for eps in eps_list:
        col_tuples.append((atk, f"{eps:.2f}"))

col_index = pd.MultiIndex.from_tuples(col_tuples, names=['Attack', 'ε'])

res_tab_bim = pd.DataFrame(index=row_index, columns=col_index, dtype=float)

print(res_tab_bim)

In [ ]:
alpha = 0.01
epsilons = [0.01, 0.02, 0.03, 0.04, 0.05, 0.075, 0.1, 0.2]

bim_attacks = [
    {
        "model_rev": model_google_gru_rev,
        "model_target": model_google_gru,
        "model_name": "GRU"
    },
    {
        "model_rev": model_google_gru_rev,
        "model_target": model_google,
        "model_name": "LSTM"
    },
    {
        "model_rev": model_google_rnn_rev,
        "model_target": model_google_rnn,
        "model_name": "RNN"
    }
]

for attack in bim_attacks:
    for eps in epsilons:
        num_iter = int(eps / alpha)

        results_bim = reverse_forecast_attack_bim(
            model_rev=attack["model_rev"],
            model_normal=attack["model_target"],
            test_loader_rev=test_loader_rev,
            test_loader=test_loader,
            epsilons=[eps],
            price_min=price_min,
            price_max=price_max,
            alpha=alpha,
            num_iter=num_iter
        )

        for eps_val, (true_vals, preds) in results_bim.items():
            log_and_plot_predictions(
                true_values=true_vals,
                predictions=preds,
                test_dates=test_dates,
                true_values_rolling=None,
                predictions_rolling=None,
                model_name=attack["model_name"],
                attack_name="REV_BIM",
                epsilon=eps_val,
                res_tab=res_tab_bim,
                similarity_fn=scalar_similarity,
                google=True,
                save_path=f"results/SEED_{seed}/plots/REV_BIM/{eps_val}",
                save_png=True
            )


In [ ]:
display(res_tab_bim)

In [ ]:
display(res_tab)

#SURROGATE PART

In [ ]:
model_google_gru_surrogate = SimpleGRU(input_size=5, hidden_size=64, output_size=1, num_layers=2)
optimizer_surrogate = torch.optim.Adam(model_google_gru_surrogate.parameters(), lr=0.001)

train_model(model_google_gru_surrogate, loss_fn, optimizer_surrogate, num_epochs, train_loader)

In [ ]:
model_google_rnn_surrogate = SimpleRNN(input_size=5, hidden_size=64, output_size=1, num_layers=2)
optimizer_surrogate = torch.optim.Adam(model_google_rnn_surrogate.parameters(), lr=0.001)

train_model(model_google_rnn_surrogate, loss_fn, optimizer_surrogate, num_epochs, train_loader)

In [ ]:
model_google_surrogate = SimpleLSTM(input_size=5, hidden_size=64, output_size=1, num_layers=2)
optimizer_surrogate = torch.optim.Adam(model_google_surrogate.parameters(), lr=0.001)

train_model(model_google_surrogate, loss_fn, optimizer_surrogate, num_epochs, train_loader)

In [ ]:
from attack.fgsm_surrogate import fgsm_surrogate_attack

In [ ]:
for eps in epsilons:
    true_vals_surro, preds_surro = fgsm_surrogate_attack(
        model_google_gru_surrogate,
        model_google_gru,
        test_loader,
        eps,
        price_min, price_max
    )

    log_and_plot_predictions(
        true_values=true_vals_surro,
        predictions=preds_surro,
        test_dates=test_dates,
        true_values_rolling=None,
        predictions_rolling=None,
        model_name='GRU',
        attack_name='FGSM_SURRO',
        epsilon=eps,
        res_tab=res_tab_bim,
        similarity_fn=scalar_similarity,
        google=True,
        save_path=f"results/SEED_{seed}/plots/FGSM_SURRO/{eps}",
        save_png=True
    )

    true_vals_surro, preds_surro = fgsm_surrogate_attack(
        model_google_rnn_surrogate,
        model_google_rnn,
        test_loader,
        eps,
        price_min, price_max
    )

    log_and_plot_predictions(
        true_values=true_vals_surro,
        predictions=preds_surro,
        test_dates=test_dates,
        true_values_rolling=None,
        predictions_rolling=None,
        model_name='RNN',
        attack_name='FGSM_SURRO',
        epsilon=eps,
        res_tab=res_tab_bim,
        similarity_fn=scalar_similarity,
        google=True,
        save_path=f"results/SEED_{seed}/plots/FGSM_SURRO/{eps}",
        save_png=True
    )

    true_vals_surro, preds_surro = fgsm_surrogate_attack(
        model_google_surrogate,
        model_google,
        test_loader,
        eps,
        price_min, price_max
    )

    log_and_plot_predictions(
        true_values=true_vals_surro,
        predictions=preds_surro,
        test_dates=test_dates,
        true_values_rolling=None,
        predictions_rolling=None,
        model_name='LSTM',
        attack_name='FGSM_SURRO',
        epsilon=eps,
        res_tab=res_tab_bim,
        similarity_fn=scalar_similarity,
        google=True,
        save_path=f"results/SEED_{seed}/plots/FGSM_SURRO/{eps}",
        save_png=True
    )

In [ ]:
display(res_tab)

In [ ]:
display(res_tab_bim)

In [ ]:
display(res_tab_fgsm)

# ANOTHER BLACK BOX ATTACK

In [ ]:
# Creation of the tab result for plotting adversial attack result
models = ['LSTM','RNN','GRU']
metrics = ['MAE', "RMSE", 'SIM']

row_index = pd.MultiIndex.from_product([models, metrics], names=['Model', 'Metric'])

attacks = ['BOUNDARY']
epsilons_attack = {
    'BOUNDARY':[0.01,0.02,0.03,0.04, 0.05, 0.075, 0.1, 0.2],
}

col_tuples = []
for atk, eps_list in epsilons_attack.items():
    for eps in eps_list:
        col_tuples.append((atk, f"{eps:.2f}"))

col_index = pd.MultiIndex.from_tuples(col_tuples, names=['Attack', 'ε'])

res_tab_boundary = pd.DataFrame(index=row_index, columns=col_index, dtype=float)

print(res_tab_boundary)

In [ ]:
from attack.boundary import boundary_attack

In [ ]:
for eps in epsilons:
  delta = eps * 0.5
  eta = eps * 0.25
  num_iter = int(20 + 200 * eps)
  true_vals_bondary, preds_bondary = boundary_attack(
      model_google_gru,
      test_loader,
      eps,
      price_min, price_max,
      num_iter,delta,eta
  )

  log_and_plot_predictions(
      true_values=true_vals_bondary,
      predictions=preds_bondary,
      test_dates=test_dates,
      true_values_rolling=None,
      predictions_rolling=None,
      model_name='GRU',
      attack_name='BOUNDARY',
      epsilon=eps,
      res_tab=res_tab_boundary,
      similarity_fn=scalar_similarity,
      google=True,
      save_path=f"results/SEED_{seed}/plots/BOUNDARY/{eps}",
      save_png=True
  )

  true_vals_bondary, preds_bondary = boundary_attack(
      model_google,
      test_loader,
      eps,
      price_min, price_max,
      num_iter,delta,eta
  )

  log_and_plot_predictions(
      true_values=true_vals_bondary,
      predictions=preds_bondary,
      test_dates=test_dates,
      true_values_rolling=None,
      predictions_rolling=None,
      model_name='LSTM',
      attack_name='BOUNDARY',
      epsilon=eps,
      res_tab=res_tab_boundary,
      similarity_fn=scalar_similarity,
      google=True,
      save_path=f"results/SEED_{seed}/plots/BOUNDARY/{eps}",
      save_png=True
  )

  true_vals_bondary, preds_bondary = boundary_attack(
      model_google_rnn,
      test_loader,
      eps,
      price_min, price_max,
      num_iter,delta,eta
  )

  log_and_plot_predictions(
      true_values=true_vals_bondary,
      predictions=preds_bondary,
      test_dates=test_dates,
      true_values_rolling=None,
      predictions_rolling=None,
      model_name='RNN',
      attack_name='BOUNDARY',
      epsilon=eps,
      res_tab=res_tab_boundary,
      similarity_fn=scalar_similarity,
      google=True,
      save_path=f"results/SEED_{seed}/plots/BOUNDARY/{eps}",
      save_png=True
  )

In [ ]:
display(res_tab_boundary)

In [ ]:
display(res_tab)

In [ ]:
result = pd.concat([res_tab, res_tab_bim,res_tab_fgsm,res_tab_boundary], axis=1)

# Affichage du résultat
display(result)
result.to_excel(f"results/SEED_{seed}/result.xlsx")